## 介紹 

本課程將涵蓋： 
- 甚麼是函數調用及其使用案例 
- 如何使用 Azure OpenAI 建立函數調用 
- 如何將函數調用整合到應用程式中 

## 學習目標 

完成本課程後，你將會知道如何以及了解： 

- 使用函數調用的目的 
- 使用 Azure Open AI 服務設定函數調用 
- 為你的應用程式使用案例設計有效的函數調用 


## 理解函數呼叫 

在這課程中，我們想為我們的教育初創公司建立一個功能，讓用戶能透過聊天機械人尋找技術課程。我們會根據用戶的技能水平、當前職位及感興趣的技術來推薦合適課程。 

為此，我們將結合使用： 
 - `Azure Open AI` 來為用戶創建聊天體驗
 - `Microsoft Learn Catalog API` 來幫助用戶根據其需求找到課程 
 - `Function Calling` 將用戶的查詢送往函數進行 API 請求。 

要開始，讓我們先了解為何我們會想要使用函數呼叫： 


### 為何使用 Function Calling  

如果你已完成本課程的其他課程，你大概已經了解使用大型語言模型（LLMs）的強大功能。希望你也能看到它們的一些限制。  

Function Calling 是 Azure Open AI 服務的一項功能，用以克服以下限制：  
1) 回應格式的一致性  
2) 能夠在聊天上下文中使用應用程式其他來源的資料  

在 Function Calling 出現之前，LLM 的回應是無結構且不一致的。開發者需要撰寫複雜的驗證程式碼，確保能處理每種不同的回應變化。  

使用者無法獲得像「斯德哥爾摩目前的天氣如何？」這樣的答案。因為模型僅限於訓練時的資料時間。  

讓我們看看下面的例子，說明這個問題：  

假設我們想建立一個學生資料庫，以便為他們建議合適的課程。下面有兩個學生描述，其包含的資料非常相似。  


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我們想將此發送給大型語言模型 (LLM) 以解析數據。這之後可以用於我們的應用程式，將其發送到 API 或存儲到數據庫中。

讓我們創建兩個相同的提示，指示 LLM 我們感興趣的資訊：


我們想將此發送至大型語言模型（LLM），以解析對我們產品重要的部分。所以我們可以建立兩個相同的提示來指示 LLM：


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


建立這兩個提示後，我們會使用 `client.responses.create` 送出給 LLM。我們將提示儲存在 `input` 變數中，並將角色指定為 `user`。這是為了模擬使用者傳送訊息給聊天機械人的情況。 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

現在我們可以同時向 LLM 發送兩個請求，並檢查我們收到的回應。


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



儘管提示相同且描述相似，我們仍然可能得到不同格式的 `Grades` 屬性。

如果你多次運行上述儲存格，格式可能是 `3.7` 或 `3.7 GPA`。

這是因為大型語言模型以書面提示的非結構化數據形式接受輸入，並且回傳的也是非結構化數據。我們需要有結構化格式，才能知道在存儲或使用這些數據時會得到什麼。

透過使用函數調用，我們可以確保收到結構化的數據。使用函數調用時，LLM 並不會真實調用或運行任何函數；相反，我們為 LLM 創建一個結構，讓它在回應時遵循。然後，我們利用這些結構化回應來知道在應用程式中該運行哪個函數。
 


![功能呼叫流程圖](../../../../translated_images/zh-HK/Function-Flow.083875364af4f4bb.webp)


我們然後可以取回函數回傳的結果並將其發送回 LLM。LLM 接著會使用自然語言回應，以回答用戶的查詢。


### 使用函數調用的使用案例

<strong>調用外部工具</strong>
聊天機械人非常擅長回應用戶的問題。透過使用函數調用，聊天機械人可以利用用戶的訊息來完成特定任務。例如，學生可以請聊天機械人「發送電郵給我的導師，說我需要這科更多的協助」。這可以呼叫一個名為 `send_email(to: string, body: string)` 的函數。


**建立 API 或資料庫查詢**
用戶可以使用自然語言找到資訊，然後轉換成格式化的查詢或 API 請求。舉例來說，一位老師請求「哪些學生完成了上一份作業」，這可以呼叫一個名為 `get_completed(student_name: string, assignment: int, current_status: string)` 的函數。


<strong>創建結構化數據</strong>
用戶可以使用 LLM 從一段文字或 CSV 中提取重要資訊。例如，學生可以將一篇關於和平協議的維基百科文章轉換，來製作 AI 記憶卡。這可以透過一個名為 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 的函數來完成。


## 2. 創建你的第一個函數呼叫 

創建函數呼叫的過程包括三個主要步驟： 
1. 使用你的函數列表和用戶訊息呼叫聊天補全 API 
2. 讀取模型的回應來執行一個動作，即執行函數或 API 呼叫 
3. 用你的函數回應，再次呼叫聊天補全 API，利用這些資訊來創建回應給用戶。 


![函數呼叫流程](../../../../translated_images/zh-HK/LLM-Flow.3285ed8caf4796d7.webp)


### 函數呼叫的元素 

#### 用戶輸入 

第一步是建立用戶訊息。這可以透過取得文字輸入的值來動態指派，或者你可以在此指派一個值。如果這是你第一次使用 Chat Completions API，我們需要定義訊息的 `role` 和 `content`。 

`role` 可以是 `system`（建立規則）、`assistant`（模型）或 `user`（最終用戶）。對於函數呼叫，我們會將這個設定為 `user` 並附上一個範例問題。 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 建立函式。 

接著我們會定義一個函式以及該函式的參數。我們只會使用一個名為 `search_courses` 的函式，但你可以建立多個函式。

<strong>重要</strong> ：函式會包含在系統訊息中傳送給 LLM，並且會計入你可用的代幣數量。 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

`name` - 我們希望調用的函數名稱。 

`description` - 這是函數如何運作的描述。在這裡清晰、具體非常重要。 

`parameters` - 你希望模型在回應中產生的值與格式清單。 


`type` - 屬性將被存放的資料類型。 

`properties` - 模型將用於回應的具體值清單。 


`name` - 模型在格式化回應中使用的屬性名稱。 

`type` - 該屬性的資料類型。 

`description` - 具體屬性的描述。 


<strong>可選</strong>

`required` - 完成函數呼叫所必需的屬性。 


### 呼叫函式
定義函式後，我們現在需要在呼叫 Chat Completion API 時將其包含進去。我們是透過在請求中加入 `functions` 來達成。在此例中為 `functions=functions`。

也有一個選項是將 `function_call` 設為 `auto`。這代表我們會讓大型語言模型根據使用者訊息決定應該呼叫哪個函式，而不是由我們自己指定。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


現在讓我們看看回應，並了解它是如何被格式化的： 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以看到函數名稱被調用，且根據用戶訊息，LLM 能找到適合函數參數的資料。 


## 3. 將函數調用整合到應用程式中。


在我們測試了來自 LLM 的格式化回應後，現在可以將其整合到應用程式中。

### 管理流程

要將此整合到我們的應用程式中，讓我們採取以下步驟：

首先，讓我們呼叫 Open AI 服務並將消息存儲在名為 `response_message` 的變數中。


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


現在我們將定義一個函數，呼叫 Microsoft Learn API 以獲取課程列表： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作為最佳實踐，我們接著會檢查模型是否想調用函數。之後，我們會創建一個可用的函數並將其與被調用的函數匹配。 
然後，我們會將該函數的參數對應到從 LLM 獲得的參數。

最後，我們會附加函數調用訊息以及由 `search_courses` 訊息返回的值。這提供了 LLM 所需的所有資訊，
以自然語言回應使用者。


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


現在我們會將更新後的訊息發送給 LLM，以便我們能收到自然語言的回應，而非 API JSON 格式的回應。


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## 代碼挑戰 

幹得好！要繼續學習 Azure Open AI 函數呼叫，你可以建立：https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 更多函數的參數，可能幫助學習者找到更多課程。你可以在這裡找到可用的 API 參數： 
 - 創建另一個函數呼叫，從學習者那裡取得更多資訊，例如他們的母語 
 - 當函數呼叫和／或 API 呼叫沒有返回任何合適課程時，建立錯誤處理 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
